In [ ]:
import requests
import pandas as pd
import os
import re

%pip install requests pandas openpyxl python-dotenv

 

In [ ]:
from dotenv import load_dotenv
import os

load_dotenv()

print(os.getenv("CANVAS_BASE_URL"))
print(os.getenv("CANVAS_TOKEN"))

In [ ]:
# 定义API的基本URL和请求头

load_dotenv()

CANVAS_BASE_URL = os.getenv("CANVAS_BASE_URL").rstrip("/")
CANVAS_TOKEN = os.getenv("CANVAS_TOKEN")

API_BASE = f"{CANVAS_BASE_URL}/api/v1"

headers = {
    "Authorization": f"Bearer {CANVAS_TOKEN}",
    "Accept": "application/json"
}

print("Canvas:", CANVAS_BASE_URL)
print("Token loaded:", bool(CANVAS_TOKEN))

In [ ]:
# 定义获取API需求的函数

def canvas_get(endpoint, params=None):
    """
    Send GET request to Canvas API and return JSON.
    """
    
    url = f"{API_BASE}{endpoint}"
    
    response = requests.get(
        url,
        headers=headers,
        params=params,
        timeout=30
    )
    
    print(f"GET {response.url}")
    print(f"Status: {response.status_code}")
    
    response.raise_for_status()
    
    return response.json()

In [ ]:
# 测试课程数据

course_id = 3308

course = canvas_get(f"/courses/{course_id}")

course["id"], course["name"]

In [33]:
# 定义2个作业和相应的rubric_id

assignments = {
    23999: {
        "name": "Assignment 1",
        "rubric_id": 1077
    },
    24000: {
        "name": "Assignment 2",
        "rubric_id": 1078
    }
}

In [51]:
# 测试assignments定义的数据

print(assignments)

{23999: {'name': 'Assignment 1', 'rubric_id': 1077}, 24000: {'name': 'Assignment 2', 'rubric_id': 1078}}


In [34]:
# 获取Rubric信息并建立criterion的字典

rubric_cache = {}

for assignment_id, info in assignments.items():
    
    rubric_id = info["rubric_id"]
    
    rubric = canvas_get(
        f"/courses/{course_id}/rubrics/{rubric_id}"
    )
    
    rubric_cache[rubric_id] = rubric

GET https://hkust-gz.instructure.com/api/v1/courses/3308/rubrics/1077
Status: 200
GET https://hkust-gz.instructure.com/api/v1/courses/3308/rubrics/1078
Status: 200


In [ ]:
# 测试输出Rubric的数据字段

rubric_cache[1077].keys()

dict_keys(['id', 'title', 'context_id', 'context_type', 'points_possible', 'reusable', 'public', 'read_only', 'free_form_criterion_comments', 'hide_score_total', 'data', 'rating_order', 'button_display'])

In [ ]:
# 测试Rubric中data字段的信息

rubric_cache[1078]["data"]

In [18]:
# 建立 criterion mapping

def build_criterion_map(rubric):
    """
    Convert Canvas rubric criteria into:
    
    criterion_id -> {
        name,
        description,
        points
    }
    """
    
    criterion_map = {}
    
    for criterion in rubric.get("data", []):
        
        criterion_id = str(criterion["id"])
        
        criterion_map[criterion_id] = {
            "name": criterion.get("description", criterion_id),
            "long_description": criterion.get("long_description"),
            "points": criterion.get("points")
        }
    
    return criterion_map

In [52]:
criterion_maps = {
    rubric_id: build_criterion_map(rubric)
    for rubric_id, rubric in rubric_cache.items()
}

print(criterion_maps.keys())

dict_keys([1077, 1078])


In [23]:
# 把 criterion 名称变成合法的 Excel 列名

def clean_column_name(name):
    """
    Make rubric criterion name suitable for DataFrame/Excel column names.
    """
    
    name = str(name).strip()
    
    # 空格 → _
    name = re.sub(r"\s+", "_", name)
    
    # 删除特殊字符
    name = re.sub(r"[^\w\u4e00-\u9fff-]", "", name)
    
    return name

In [24]:
def build_column_map(criterion_map):
    
    column_map = {}
    
    for criterion_id, info in criterion_map.items():
        
        criterion_name = clean_column_name(info["name"])
        
        column_map[criterion_id] = {
            "score": f"{criterion_name}_score",
            "comment": f"{criterion_name}_comment"
        }
    
    return column_map

In [53]:
column_maps = {
    rubric_id: build_column_map(criterion_map)
    for rubric_id, criterion_map in criterion_maps.items()
}

# 测试column_maps的输出
print(column_maps.keys())

dict_keys([1077, 1078])


In [54]:
# 获取 Peer Review 原始数据

peer_raw = {}

for assignment_id, info in assignments.items():
    
    rubric_id = info["rubric_id"]
    
    data = canvas_get(
        f"/courses/{course_id}/rubrics/{rubric_id}",
        params={
            "include[]": "peer_assessments",
            "style": "full"
        }
    )
    
    peer_raw[assignment_id] = data

GET https://hkust-gz.instructure.com/api/v1/courses/3308/rubrics/1077?include%5B%5D=peer_assessments&style=full
Status: 200
GET https://hkust-gz.instructure.com/api/v1/courses/3308/rubrics/1078?include%5B%5D=peer_assessments&style=full
Status: 200


In [ ]:
# 测试peer_raw输出

peer_raw[23999]["assessments"][:2]

In [56]:
# 获取peer review -> student的对应关系

peer_review_map = {}

for assignment_id in assignments:
    
    peer_reviews = canvas_get(
        f"/courses/{course_id}/assignments/{assignment_id}/peer_reviews"
    )
    
    peer_review_map[assignment_id] = peer_reviews

GET https://hkust-gz.instructure.com/api/v1/courses/3308/assignments/23999/peer_reviews
Status: 200
GET https://hkust-gz.instructure.com/api/v1/courses/3308/assignments/24000/peer_reviews
Status: 200


In [66]:
# 测试输出peer_review_map的前2条记录

peer_review_map[24000][:2]

[{'id': 6880,
  'user_id': 393,
  'assessor_id': 14395,
  'asset_id': 742935,
  'asset_type': 'Submission',
  'workflow_state': 'completed'},
 {'id': 6905,
  'user_id': 393,
  'assessor_id': 10781,
  'asset_id': 742935,
  'asset_type': 'Submission',
  'workflow_state': 'completed'}]

In [58]:
# 建立 artifact -> student 的对应关系

artifact_to_student = {}

for assignment_id, peer_reviews in peer_review_map.items():
    
    artifact_to_student[assignment_id] = {}
    
    for review in peer_reviews:
        
        artifact_id = review.get("asset_id")
        student_id = review.get("user_id")
        
        artifact_to_student[assignment_id][artifact_id] = student_id

In [ ]:
# 测试

artifact_to_student[24000]

In [62]:
# 把 Peer Review 转成最终表格的一行一行

peer_rows = []

for assignment_id, info in assignments.items():
    
    rubric_id = info["rubric_id"]
    
    rubric = peer_raw[assignment_id]
    
    criterion_map = criterion_maps[rubric_id]
    column_map = column_maps[rubric_id]
    
    assessments = rubric.get("assessments", [])
    
    for assessment in assessments:
        
        # 只保留 peer review
        if assessment.get("assessment_type") != "peer_review":
            continue
        
        artifact_id = assessment.get("artifact_id")
        assessor_id = assessment.get("assessor_id")
        
        # artifact → student
        student_id = artifact_to_student[assignment_id].get(
            artifact_id
        )
        
        # 先建立这一位 reviewer 的一整行
        row = {
            "student_id": student_id,
            "assignment_id": assignment_id,
            "assessment_type": "peer_review",
            "assessor_id": assessor_id,
            "assessment_id": assessment.get("id"),
            "artifact_id": artifact_id,
            "total_score": assessment.get("score")
        }
        
        # 再把每个 criterion 放进去
        for criterion in assessment.get("data", []):
            
            criterion_id = str(
                criterion.get("criterion_id")
            )
            
            if criterion_id not in column_map:
                continue
            
            score_column = column_map[criterion_id]["score"]
            comment_column = column_map[criterion_id]["comment"]
            
            row[score_column] = criterion.get("points")
            row[comment_column] = criterion.get("comments")
        
        peer_rows.append(row)

In [64]:
# 测试

peer_df = pd.DataFrame(peer_rows)

In [ ]:
# 查看 Peer review DataFrame 中 assignment id = 23999 的数据，前10行+前17列

peer_df.iloc[:10, :17]

In [ ]:
# 查看 Peer review DataFrame 的前面assignment id = 24000 的数据，末尾10行+末尾17列

peer_df.iloc[-20:, -10:]

In [93]:
# 处理 TA Grading

submission_raw = {}

for assignment_id in assignments:
    
    submissions = canvas_get(
        f"/courses/{course_id}/assignments/{assignment_id}/submissions",
        params={
            "per_page": 100
        }
    )
    
    submission_raw[assignment_id] = submissions

GET https://hkust-gz.instructure.com/api/v1/courses/3308/assignments/23999/submissions?per_page=100
Status: 200
GET https://hkust-gz.instructure.com/api/v1/courses/3308/assignments/24000/submissions?per_page=100
Status: 200


In [95]:
# 解析TA grading

ta_rows = []

for assignment_id, submissions in submission_raw.items():
    
    rubric_id = assignments[assignment_id]["rubric_id"]
    
    criterion_map = criterion_maps[rubric_id]
    column_map = column_maps[rubric_id]
    
    for submission in submissions:
        
        student_id = submission.get("user_id")
        
        rubric_assessment = submission.get(
            "rubric_assessment"
        )
        
        # 如果这里没有 rubric assessment，
        # 说明这个学生没有 rubric grading
        if not rubric_assessment:
            continue
        
        row = {
            "student_id": student_id,
            "assignment_id": assignment_id,
            "assessment_type": "grading",
            "assessor_id": submission.get("grader_id"),
            "assessment_id": None,
            "artifact_id": submission.get("id"),
            "total_score": None
        }
        
        total_score = 0
        
        for criterion_id, criterion_data in rubric_assessment.items():
            
            criterion_id = str(criterion_id)
            
            if criterion_id not in column_map:
                continue
            
            points = criterion_data.get("points")
            
            score_column = column_map[criterion_id]["score"]
            comment_column = column_map[criterion_id]["comment"]
            
            row[score_column] = points
            row[comment_column] = criterion_data.get(
                "comments"
            )
            
            # 用 criterion points 求 rubric 总分
            if points is not None:
                total_score += points
        
        row["total_score"] = total_score
        
        ta_rows.append(row)

In [98]:
# 转化成dataframe，并查看TA grading 的数据

ta_df = pd.DataFrame(ta_rows)
ta_df.head()

""
